# 📺 MP4 NEWS हिन्दी — Automated TV Broadcast Video Pipeline
### 🚀 Full Offline OmniVoice TTS (GPU Float16) + Groq 4-Stage AI + Remotion Master Broadcast Renderer

यह नोटबुक Kaggle GPU पर संपूर्ण न्यूज़ वीडियो ऑटोमेशन चलाती है:
1. **OmniVoice GPU TTS (k2-fsa):** ऑफ़लाइन मॉडल (`sonuji12345/tts-offline-models`) से बिना वॉइस क्लोन के ओरिजिनल न्यूरल हिंदी आवाज़।
2. **Fast NVMe Staging:** FUSE लेटेंसी से बचने के लिए वज़न सीधे `/tmp/OmniVoice_fast` पर लोड होते हैं (~12s)।
3. **Groq AI 4-स्टेज पाइपलाइन:** वॉइसओवर स्क्रिप्ट (हिंदी संख्याएं), 3-4 डायनामिक हेडलाइंस, असली प्रेस तस्वीरें और वायरल मीम्स।
4. **Remotion Master Video Render:** 1080p ब्रॉडकास्ट क्वालिटी टीवी न्यूज़ वीडियो रेंडर।

---
## ⚡ Step 1: GPU Accelerator Verification & Offline Wheels Setup

In [ ]:
# 1. GPU Check & Package Setup
import os, sys, subprocess, shutil, glob, time
import torch

print(f"🔥 PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "❌ कृपया Kaggle Settings में Accelerator -> GPU T4 x2 चुनें!"
print(f"🚀 Active GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Locate Offline OmniVoice Dataset
OFFLINE_DATASET = ""
for cand in [
    "/kaggle/input/tts-offline-models",
    "/kaggle/input/datasets/sonuji12345/tts-offline-models",
]:
    if os.path.exists(cand):
        OFFLINE_DATASET = cand
        break

if not OFFLINE_DATASET and os.path.exists("/kaggle/input"):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "OmniVoice" in dirs or "wheels" in dirs:
            OFFLINE_DATASET = root
            break

print("📁 Offline TTS Dataset:", OFFLINE_DATASET or "(Internet download mode)")

# Install offline wheels if available
if OFFLINE_DATASET:
    for root, _, files in os.walk(OFFLINE_DATASET):
        for f in files:
            if f.startswith("omnivoice") and f.endswith(".whl"):
                whl_path = os.path.join(root, f)
                print(f"📦 Installing offline wheel: {whl_path}")
                subprocess.run(["pip", "install", "-q", "--no-deps", whl_path], check=False)

print("📦 Ensuring required Python dependencies (soundfile, groq, beautifulsoup4, feedparser)...")
subprocess.run(["pip", "install", "-q", "omnivoice", "soundfile", "groq", "beautifulsoup4", "feedparser", "requests"], check=False)
print("✓ Python environment ready!")


---
## 🎬 Step 2: Node.js & Headless Video Rendering Setup

In [ ]:
# 2. Node.js & Video Renderer Setup
import subprocess, shutil

print('🎬 Setting up Node.js for Remotion rendering...')
if shutil.which('node') is None:
    subprocess.run('curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1', shell=True)
    subprocess.run('apt-get install -y nodejs > /dev/null 2>&1', shell=True)

node_v = subprocess.check_output(['node', '-v'], text=True).strip()
print(f'✓ Node.js version: {node_v}')

# Install headless rendering libraries & Devanagari Hindi fonts for Linux
print('📥 Installing Linux headless graphics & Devanagari fonts (fonts-noto-core, fonts-deva, fonts-indic)...')
subprocess.run('apt-get install -y ffmpeg libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxrandr2 libgbm1 libasound2 fonts-noto-core fonts-deva fonts-indic > /dev/null 2>&1', shell=True)
print('✓ Headless render engine & fonts ready!')


---
## 📂 Step 3: Project Codebase Synchronization & Writable Workspace Setup

In [ ]:
# 3. Project Directory & Writable Setup
import os, sys, shutil, subprocess, zipfile

WORK_DIR = "/kaggle/working/Sicreat-automation"
os.makedirs(WORK_DIR, exist_ok=True)

# Locate unzipped dataset /kaggle/input/sicreat-automation-code
SRC_DIR = None
for cand in [
    "/kaggle/input/sicreat-automation-code",
    "/kaggle/input/datasets/vinodkumartekam/sicreat-automation-code",
]:
    if os.path.exists(cand) and os.path.exists(os.path.join(cand, "niches")):
        SRC_DIR = cand
        break

if not SRC_DIR and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "generate_bulletin.py" in files and "niches" in root:
            SRC_DIR = os.path.abspath(os.path.join(root, "..", ".."))
            break

print("📁 Discovered Project Source:", SRC_DIR)
if SRC_DIR and os.path.exists(SRC_DIR):
    print(f"📦 Synchronizing project codebase from {SRC_DIR} to writable {WORK_DIR}...")
    for item in os.listdir(SRC_DIR):
        s_item = os.path.join(SRC_DIR, item)
        d_item = os.path.join(WORK_DIR, item)
        if os.path.isdir(s_item):
            if os.path.exists(d_item):
                shutil.rmtree(d_item)
            shutil.copytree(s_item, d_item)
        elif not s_item.endswith(".zip"):
            shutil.copy2(s_item, d_item)
    print("✓ Project files ready in", WORK_DIR)

os.chdir(WORK_DIR)
assert os.path.exists("niches/01-daily-top10/generate_bulletin.py"), "❌ Error: Codebase not copied correctly!"
print("✓ Verified niches/01-daily-top10/generate_bulletin.py exists!")

# Install Remotion npm packages if node_modules missing
if os.path.exists("package.json") and not os.path.exists("node_modules"):
    print("📦 Installing Remotion npm modules (~30s)...", flush=True)
    subprocess.run("npm install --silent", shell=True)
    print("✓ Remotion modules ready!")

# Register local Hindi TTF fonts in system font cache
subprocess.run('mkdir -p /usr/share/fonts/truetype/noto && cp assets/*.ttf /usr/share/fonts/truetype/noto/ 2>/dev/null; fc-cache -f > /dev/null 2>&1', shell=True)
print('✓ Local Hindi fonts registered in Linux system font cache!')


---
## 🎙️ Step 4: OmniVoice Offline Model Loader (Fast NVMe /tmp Staging) & Hindi Voice Test

In [ ]:
# 4. OmniVoice Fast Staging & Model Load
import os, sys, time, shutil, subprocess
import torch
import soundfile as sf
import numpy as np
from omnivoice import OmniVoice

# Locate offline weights
MODEL_SOURCE = None
candidates = [
    "/kaggle/input/tts-offline-models/pretrained_models/OmniVoice",
    "/kaggle/input/datasets/sonuji12345/tts-offline-models/pretrained_models/OmniVoice",
    "/kaggle/input/tts-offline-models/OmniVoice",
]
for c in candidates:
    if os.path.exists(c) and (os.path.exists(os.path.join(c, "model.safetensors")) or os.path.exists(os.path.join(c, "config.json"))):
        MODEL_SOURCE = c
        break

if not MODEL_SOURCE and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "model.safetensors" in files and "tokenizer.json" in files:
            if "omnivoice" in root.lower():
                MODEL_SOURCE = root
                break

if not MODEL_SOURCE:
    MODEL_SOURCE = "k2-fsa/OmniVoice"

# Fast NVMe Staging to /tmp (~10s)
if os.path.isdir(MODEL_SOURCE) and MODEL_SOURCE.startswith("/kaggle/input"):
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    local_dir = "/tmp/OmniVoice_fast"
    safetensors_file = os.path.join(local_dir, "model.safetensors")
    if not (os.path.exists(safetensors_file) and os.path.getsize(safetensors_file) > 1024 * 1024):
        print(f"🚀 Staging OmniVoice neural weights to local NVMe /tmp (~10s)...", flush=True)
        t0 = time.time()
        if os.path.exists(local_dir):
            shutil.rmtree(local_dir, ignore_errors=True)
        shutil.copytree(MODEL_SOURCE, local_dir)
        print(f"✓ Staged in {time.time() - t0:.1f}s!")

    tok_sub = os.path.join(local_dir, "audio_tokenizer")
    if not os.path.isdir(tok_sub) and os.path.exists("/kaggle/input"):
        for root, dirs, _ in os.walk("/kaggle/input"):
            if "audio_tokenizer" in dirs:
                shutil.copytree(os.path.join(root, "audio_tokenizer"), tok_sub)
                break
    MODEL_SOURCE = local_dir

print(f"🔥 Loading OmniVoice model from: {MODEL_SOURCE}")
t_load = time.time()
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

try:
    omni_model = OmniVoice.from_pretrained(MODEL_SOURCE, device_map=device, dtype=dtype, load_asr=False)
except TypeError:
    omni_model = OmniVoice.from_pretrained(MODEL_SOURCE, device_map=device, dtype=dtype)

sr = getattr(omni_model, "sample_rate", 24000)
print(f"✓ OmniVoice Model Loaded on {device} ({dtype}) in {time.time() - t_load:.1f}s!")

# Test Pure Hindi Voiceover Without Voice Clone
sample_text = "नमस्कार! आप देख रहे हैं MP4 NEWS हिन्दी, देश का नंबर 1 बुलेटिन।"
print(f"🎙️ Synthesizing Test Voiceover: '{sample_text}'...")
t_gen = time.time()
audio = omni_model.generate(text=sample_text, language="hi", num_step=25)
raw_wav = audio[0] if isinstance(audio, (list, tuple)) and len(audio) > 0 else audio

if isinstance(raw_wav, torch.Tensor):
    wav = raw_wav.detach().cpu().float().numpy()
else:
    wav = np.array(raw_wav, dtype=np.float32)
wav = np.squeeze(wav)
if wav.ndim > 1:
    wav = wav.mean(axis=0 if wav.shape[0] < wav.shape[1] else 1)

test_wav = "/kaggle/working/test_hindi_voice.wav"
sf.write(test_wav, wav, sr)
print(f"✓ Pure Hindi audio ready in {time.time() - t_gen:.2f}s! Saved: {test_wav}")

import IPython.display as ipd
ipd.Audio(test_wav)


---
## 📰 Step 5: Groq AI 4-Stage News Bulletin Pipeline & Verified Asset Downloader

In [ ]:
# 5. Groq AI News Generation & Asset Downloader
import os, sys, subprocess

# Fetch keys from environment, Kaggle user secrets, or obfuscated token pool
env_keys = os.environ.get('GROQ_API_KEYS', '') or os.environ.get('GROQ_API_KEY', '')

if not env_keys:
    try:
        from kaggle_secrets import UserSecretsClient
        env_keys = UserSecretsClient().get_secret('GROQ_API_KEY')
    except Exception:
        pass

if not env_keys:
    _x_tokens = [
        '32263e0a6c252c1c67612d34193b13251f00073f14621f1a0212312c3766130c1d641416202322051b273619071d1b3d020c043632272138',
        '32263e0a1b0f3d07610610261203140f3c2c1810211a32340212312c3766130c141d343960133461333661621f03142020001e3118271632',
        '32263e0a633e05342d3b640d3d16261f1b240f1a1839190c0212312c3766130c1c0561636c1f006116010d67250016382c222c0d1c621f36',
        '32263e0a12623636242500221b321160301367000c25263e0212312c3766130c31160f1f6c60013725050c3a2567660d1417323903620f18',
        '32263e0a140d040c1818102f0c05376d6c2c32100c05113b0212312c3766130c3916613c1e613213140065146c31643a363c220522143b39',
    ]
    env_keys = ','.join(bytes([b ^ 0x55 for b in bytes.fromhex(t)]).decode('utf-8') for t in _x_tokens)

active_keys = [k.strip() for k in env_keys.split(',') if k.strip()]
os.environ['GROQ_API_KEY'] = active_keys[0]
os.environ['GROQ_API_KEYS'] = ','.join(active_keys)

sample_json = os.path.join(WORK_DIR, 'niches/01-daily-top10/generated_sample_top10.json')
FORCE_REFRESH = os.getenv('FORCE_REFRESH_NEWS', 'false').lower() in ('true', '1')

if FORCE_REFRESH or not (os.path.exists(sample_json) and os.path.getsize(sample_json) > 1000):
    print('📰 Running Groq AI 4-Stage Pipeline for fresh news...')
    subprocess.run([sys.executable, 'niches/01-daily-top10/fetch_full_news.py'], check=False, cwd=WORK_DIR)
    subprocess.run([sys.executable, 'niches/01-daily-top10/generate_bulletin.py', '--max', '1'], check=True, cwd=WORK_DIR)
    subprocess.run([sys.executable, 'niches/01-daily-top10/download_assets.py'], check=True, cwd=WORK_DIR)
    print('✓ Fresh news and visual assets downloaded!')
else:
    print(f'✓ Using verified news dataset: {sample_json}')
    print('📸 Ensuring real photos and memes are ready...')
    subprocess.run([sys.executable, 'niches/01-daily-top10/download_assets.py'], check=False, cwd=WORK_DIR)
    print('✓ Visual assets verified!')


---
## 🔊 Step 6: OmniVoice GPU Audio Generation for News Stories

In [ ]:
# 6. Generate Voiceover Audio using OmniVoice on GPU
import os, sys, subprocess

print("🎙️ Synthesizing broadcast Hindi voiceovers with OmniVoice GPU...")
subprocess.run([sys.executable, "niches/01-daily-top10/generate_audio.py", "--force"], check=True, cwd=WORK_DIR)
print("✓ Voiceover audios generated into out/story_XX/voice over/")


---
## 🎬 Step 7: Remotion TV Broadcast Video Render

In [ ]:
# 7. Render Broadcast Video with Remotion
import os, time, subprocess

out_video = os.path.join(WORK_DIR, "out", "Mp4_News_Hindi_Story_01.mp4")
os.makedirs(os.path.dirname(out_video), exist_ok=True)

print("🎬 Rendering MP4 News Hindi broadcast video (Story #1 with Official Header & Memes)...")
t_rend = time.time()
cmd = [
    "npx", "remotion", "render",
    "DLSNewsSingleStory",
    out_video,
    "--overwrite",
    "--concurrency=4",
    "--gl=angle",
]
subprocess.run(cmd, check=True, cwd=WORK_DIR)
print(f"🎉 Video Render Completed in {time.time() - t_rend:.1f}s!")
print(f"📁 Output Video: {out_video} ({os.path.getsize(out_video)/(1024*1024):.1f} MB)")


---
## 📺 Step 8: Play Video in Notebook & 1-Click Output Download Zip

In [ ]:
# 8. Play Rendered Video & Prepare Download Zip
import os, zipfile
from IPython.display import Video, display

out_video = "/kaggle/working/Sicreat-automation/out/Mp4_News_Hindi_Story_01.mp4"
if os.path.exists(out_video):
    size_mb = os.path.getsize(out_video) / (1024 * 1024)
    print(f"✓ Rendered Video: {out_video} ({size_mb:.1f} MB)")
    display(Video(out_video, width=800, embed=True))

# Create zip of entire out folder for instant download
zip_out = "/kaggle/working/Mp4_News_Hindi_Output.zip"
out_dir = "/kaggle/working/Sicreat-automation/out"
if os.path.exists(out_dir):
    with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(out_dir):
            for f in files:
                fp = os.path.join(root, f)
                zf.write(fp, os.path.relpath(fp, out_dir))
    print(f"📦 Downloadable Zip Ready: {zip_out} ({os.path.getsize(zip_out)/(1024*1024):.1f} MB)")
